In [ ]:
from graph import extract_triples_propositions, extract_propositions
from display_graph import plot_knowledge_graph
from concept import ConceptGraph, BiEncoderSimilarity, CrossEncoderSimilarity
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Generation Graphe

Faire un forward depuis P sur toutes les relations (ou presque)
Depuis H, faire un forward sur des relations comme "HasPrerequisite"

In [2]:
P = "He eats chicken."
H = "He eats meat."
## Tester P = "He woke up at 3a.m." et H = "He woke up at 3p.m." (relation pm, Antonym, am)
g_p = extract_triples_propositions(P)
g_h = extract_triples_propositions(H)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [ ]:
plot_knowledge_graph(g_p, text = P)

In [ ]:
plot_knowledge_graph(g_h, text = H)

## Test recuperer des données

In [3]:
ConceptGraph = ConceptGraph(lang='en')
## ConceptGraph.get_rel_forward('ability')

In [4]:
g_p

{('He', 'PerformsAction', 'eating'), ('chicken', 'ReceivesAction', 'eating')}

In [5]:
data_p = ConceptGraph.get_rel_from_triples(g_p, dist = 1)
data_p[:8], data_p.shape

(array([['chicken', 'RelatedTo', 'bird', 10.768],
        ['eating', 'RelatedTo', 'plate', 9.978],
        ['chicken', 'RelatedTo', 'egg', 9.406],
        ['chicken', 'RelatedTo', 'hen', 8.103],
        ['chicken', 'RelatedTo', 'rooster', 7.657],
        ['chicken', 'RelatedTo', 'animal', 7.6],
        ['chicken', 'CapableOf', 'cross_road', 7.483],
        ['chicken', 'IsA', 'food', 7.211]], dtype=object),
 (90, 4))

Les relations avec le score le plus elevé ne sont pas forcement les plus pertinentes pour lier P à H.

### Bi-Encoder

In [6]:
bi_encoder_similarity = BiEncoderSimilarity()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
bi_encoder_similarity.compare_sentence_triples(H, data_p)[:10]
# On compare les triplets avec l'hypothese

array([['chicken', 'RelatedTo', 'meat', 5.759, 0.5358201861381531],
       ['chicken', 'IsA', 'meat', 6.633, 0.519403874874115],
       ['chicken', 'RelatedTo', 'eating', 3.394, 0.5092950463294983],
       ['chicken', 'RelatedTo', 'white_meat', 2.542, 0.48845338821411133],
       ['eating', 'RelatedTo', 'squirrel', 2.979, 0.4655942916870117],
       ['eating', 'RelatedTo', 'chicken', 3.394, 0.4629976153373718],
       ['chicken', 'RelatedTo', 'food', 6.887, 0.44427675008773804],
       ['chicken', 'IsA', 'food', 7.211, 0.4326719045639038],
       ['chicken', 'RelatedTo', 'animal', 7.6, 0.42774707078933716],
       ['eating', 'HasPrerequisite', 'food', 2.828, 0.41815289855003357]],
      dtype=object)

In [8]:
bi_encoder_similarity.compare_sentence_triples(P, data_p)[:15]
# On compare les triplets avec la prémisse

array([['chicken', 'RelatedTo', 'eating', 3.394, 0.7187103629112244],
       ['eating', 'RelatedTo', 'chicken', 3.394, 0.6627695560455322],
       ['chicken', 'RelatedTo', 'food', 6.887, 0.6617285013198853],
       ['chicken', 'RelatedTo', 'dinner', 3.007, 0.6231511831283569],
       ['chicken', 'RelatedTo', 'edible', 3.076, 0.6161932945251465],
       ['chicken', 'RelatedTo', 'poultry', 6.727, 0.6086997985839844],
       ['chicken', 'RelatedTo', 'meat', 5.759, 0.6086231470108032],
       ['chicken/n/wn/food', 'Synonym', 'poulet/n/wn/food', 2.0,
        0.6040293574333191],
       ['chicken', 'RelatedTo', 'beak', 3.118, 0.5997651815414429],
       ['chicken', 'RelatedTo', 'animal', 7.6, 0.5992805361747742],
       ['chicken', 'RelatedTo', 'bird', 10.768, 0.599157452583313],
       ['chicken/n/wn/food', 'PartOf', 'chicken/n/wn/animal', 2.0,
        0.5830645561218262],
       ['chicken', 'RelatedTo', 'tasty', 2.253, 0.5796024203300476],
       ['chicken', 'RelatedTo', 'like', 2.457, 0.5

In [9]:
# On test sur un ensemble de triplet avec une distance de 2 en comparant avec l'hypothese
bi_encoder_similarity.compare_sentence_triples(H, ConceptGraph.get_rel_from_triples(g_p, dist = 2))[:10]

array([['meat', 'UsedFor', 'eating', 3.464, 0.705134391784668],
       ['meat', 'UsedFor', 'eat', 4.472, 0.6937941312789917],
       ['meat', 'UsedFor', 'meals', 2.828, 0.6340715885162354],
       ['meat', 'IsA', 'flesh', 2.0, 0.5907585620880127],
       ['meat', 'UsedFor', 'people_to_eat', 2.0, 0.5824645757675171],
       ['meat', 'RelatedTo', 'butcher', 3.755, 0.5598814487457275],
       ['animal', 'CapableOf', 'eating', 3.464, 0.5576512813568115],
       ['meat', 'RelatedTo', 'chicken', 5.759, 0.5477034449577332],
       ['chicken', 'RelatedTo', 'meat', 5.759, 0.5358201265335083],
       ['meat', 'IsA', 'not_bone', 2.0, 0.5327609777450562]], dtype=object)

On voit qu'en comparant les triplets ConceptNET avec H, on obtient en premier un lien avec 'meat', le mot centrale de l'hypothese, alors qu'en comparant avec P, 'meat' n'apparrait pas parmis les premiers. Cependant en comparant avec H, la premiere relation est 'RelatedTo' et non 'IsA', 'RelatedTo' n'apporte pas vraiment d'information pertinente, on voudrait que 'IsA' soit la premiere. Mais quand on met une distance de 2, tous les premiers lien sont entre "meat" et "eating" (car le premier lien s'est fait de chicken vers meat, ce qui permet le deuxieme lien entre meat et eating. Car la relation qui les lie (UsedFor) n'est pas bi-directionnelle, et donc on ne peut partir que de meat, et non de eating). Il faudrait un mécanisme d'attention pour voir sur quels termes se concentrer.

Le but est de faire un pont entre P et H, mais un pont qui permet d'inferer et pas juste de completer. Le modele doit prioritairement lier chicken avec meat, plutot que meat avec eating.

Solutions : Cross-Encoder plutot que Bi-Encoder.

### Cross-Encoder

In [10]:
# On va tester sur "cross-encoder/nli-deberta-v3-base"
cross_encoder_similarity = CrossEncoderSimilarity()

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [11]:
cross_encoder_similarity.compare_sentence_triples(H, data_p)[:10]


array([['chicken', 'RelatedTo', 'meat', 5.759, -5.684865951538086],
       ['chicken', 'IsA', 'meat', 6.633, -6.086508750915527],
       ['eating', 'HasPrerequisite', 'food', 2.828, -6.727169036865234],
       ['chicken', 'RelatedTo', 'eating', 3.394, -6.965595245361328],
       ['chicken', 'RelatedTo', 'white_meat', 2.542, -7.026357650756836],
       ['eating', 'RelatedTo', 'chicken', 3.394, -7.292215347290039],
       ['eating', 'HasSubevent', 'chewing', 7.211, -7.569958686828613],
       ['eating', 'HasSubevent', 'swallow', 2.828, -7.75788688659668],
       ['eating', 'HasSubevent', 'bite', 2.0, -7.787076950073242],
       ['eating', 'HasSubevent', 'chew', 2.0, -7.878334999084473]],
      dtype=object)

In [12]:
cross_encoder_similarity.compare_sentence_triples(H, ConceptGraph.get_rel_from_triples(g_p, dist = 2))[:25]

array([['meat', 'UsedFor', 'eat', 4.472, -0.3654164671897888],
       ['meat', 'UsedFor', 'eating', 3.464, -1.1236891746520996],
       ['meat', 'UsedFor', 'people_to_eat', 2.0, -2.784234046936035],
       ['meat', 'IsA', 'flesh', 2.0, -3.512795925140381],
       ['meat', 'UsedFor', 'meals', 2.828, -3.6178970336914062],
       ['meat', 'RelatedTo', 'butcher', 3.755, -4.351190567016602],
       ['meat', 'IsA', 'not_bone', 2.0, -4.89321231842041],
       ['animal', 'Desires', 'eat', 4.0, -5.2747392654418945],
       ['meat', 'RelatedTo', 'steak', 2.964, -5.297188758850098],
       ['meat', 'IsA', 'animal_muscles', 2.0, -5.408056735992432],
       ['meat', 'RelatedTo', 'chicken', 5.759, -5.534515380859375],
       ['chicken', 'RelatedTo', 'meat', 5.759, -5.684865951538086],
       ['meat/n/wn/food', 'IsA', 'food/n/wn/food', 2.0,
        -6.078830242156982],
       ['chicken', 'IsA', 'meat', 6.633, -6.086508750915527],
       ['squirrel', 'RelatedTo', 'eats', 4.482, -6.172757148742676],
  

Meme probleme que pour le bi-encoder.
Sinon : A chaque iteration, compute les chemin d'une distance = 1, ajouter ceux qui ont le meilleur score de similarité avec H, puis ensuite repartir des nouveau noeud et recommencer le compute